# Day 1-03｜Roboflow Court Keypoint Dataset 與 YOLO Pose 訓練
> Python 籃球運動資料分析課程  
> 本單元建立球場 keypoint 標註、Roboflow 匯出格式、COCO-to-YOLO Pose 轉換與 Ultralytics 訓練流程。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 確認球場 keypoint dataset 的標註格式與固定點位順序。
- 將 Roboflow COCO keypoint export 轉為 Ultralytics YOLO pose dataset。
- 讀取教師提供的 court keypoint model，對實際籃球影片 frame 執行推論。

## 資料放置位置
- Roboflow COCO keypoint export：`assets/datasets/roboflow_court_coco/`
- 轉換後 YOLO pose dataset：`assets/datasets/roboflow_court_yolo_pose/`
- 已訓練 court keypoint 權重：`assets/models/court_keypoints/yolo26n_basketball_court_pose_best.pt`


## 課程流程
1. 檢查 Roboflow keypoint dataset 放置位置。
2. 必要時把 COCO keypoint export 轉成 YOLO pose 格式。
3. 閱讀實際訓練程式；預設不在課堂重新訓練。
4. 使用已訓練模型對參考影片 frame 執行 court keypoint 推論。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


## Roboflow Keypoint Export 格式
Roboflow COCO keypoint export 預期結構如下。`_annotations.coco.json` 需包含 `categories[].keypoints`、`categories[].skeleton` 與每筆 annotation 的 `keypoints` 欄位。

```text
assets/datasets/roboflow_court_coco/
├── train/
│   ├── _annotations.coco.json
│   └── *.jpg
├── valid/
│   ├── _annotations.coco.json
│   └── *.jpg
└── test/
    ├── _annotations.coco.json
    └── *.jpg
```

轉換後的 Ultralytics YOLO pose dataset 會放在 `assets/datasets/roboflow_court_yolo_pose/`，其中 `data.yaml` 會包含 `kpt_shape`、`flip_idx`、`keypoint_names` 與 `keypoint_skeleton`。


In [ ]:
from src.roboflow_utils import dataset_status

COURT_COCO_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_coco"
COURT_YOLO_POSE_DIR = COURSE_ROOT / "assets" / "datasets" / "roboflow_court_yolo_pose"
COURT_MODEL_PATH = (
    COURSE_ROOT
    / "assets"
    / "models"
    / "court_keypoints"
    / "yolo26n_basketball_court_pose_best.pt"
)

print("COCO keypoint export:")
print(dataset_status(COURT_COCO_DIR))
print("\nYOLO pose dataset:")
print(dataset_status(COURT_YOLO_POSE_DIR))
print("\nmodel exists:", COURT_MODEL_PATH.exists(), COURT_MODEL_PATH)


In [ ]:
from src.roboflow_utils import convert_roboflow_coco_keypoints_to_yolo_pose, find_coco_annotation

RUN_CONVERSION = False

if RUN_CONVERSION:
    if find_coco_annotation(COURT_COCO_DIR / "train") is None:
        raise FileNotFoundError(
            "請先將 Roboflow COCO keypoint export 放到 assets/datasets/roboflow_court_coco/"
        )
    data_yaml = convert_roboflow_coco_keypoints_to_yolo_pose(
        COURT_COCO_DIR,
        COURT_YOLO_POSE_DIR,
        overwrite=True,
    )
    print("converted data.yaml:", data_yaml)
else:
    print("RUN_CONVERSION=False；若已放入 Roboflow COCO keypoint export，可改為 True 執行轉換。")


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    from ultralytics import YOLO

    data_yaml = COURT_YOLO_POSE_DIR / "data.yaml"
    if not data_yaml.exists():
        raise FileNotFoundError(
            "找不到 YOLO pose data.yaml。請先執行 COCO-to-YOLO Pose 轉換。"
        )

    model = YOLO("yolo26n-pose.pt")
    results = model.train(
        data=str(data_yaml),
        epochs=200,
        imgsz=960,
        batch=-1,
        workers=8,
        patience=50,
        project=str(COURSE_ROOT / "assets" / "results" / "training" / "court_keypoints"),
        name="yolo26n_basketball_court_pose",
        fliplr=0.0,
        plots=True,
    )
    print(results)
else:
    print("RUN_TRAINING=False；本課程預設使用 assets/models/court_keypoints/ 內的已訓練權重。")


In [ ]:
import pandas as pd

from src.cv_utils import draw_points, save_image_rgb, show_image, save_json
from src.geometry_utils import render_bev_court
from src.yolo_utils import (
    court_keypoint_model_path,
    first_reference_video,
    read_video_frame,
    records_to_dicts,
    run_court_keypoints_on_image,
)

video_path = first_reference_video(COURSE_ROOT)
frame = read_video_frame(video_path, frame_index=15)
bev = render_bev_court(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json")

keypoints, H, _ = run_court_keypoints_on_image(
    court_keypoint_model_path(COURSE_ROOT),
    frame,
    bev.shape,
    conf=0.25,
    anchor_confidence=0.35,
    imgsz=960,
    frame_index=15,
)

points = [kp.image_xy for kp in keypoints]
labels = [f"{kp.label} {kp.confidence:.2f}" for kp in keypoints]
vis = draw_points(frame, points, labels=labels)
show_image(vis, "court keypoints on reference frame")

out_image = COURSE_ROOT / "assets" / "results" / "d1_03_court_keypoints.png"
out_json = COURSE_ROOT / "assets" / "results" / "d1_03_court_keypoints.json"
save_image_rgb(out_image, vis)
save_json(records_to_dicts(keypoints), out_json)

print("video:", video_path)
print("keypoint count:", len(keypoints))
print("homography estimated:", H is not None)
print("saved:", out_image)
pd.DataFrame(records_to_dicts(keypoints)).head()
